# 第17周 · 作业：研究实验 4——FM vs DDPM 受控对比 · guidance 强度的权衡（预计约7小时）

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 研究实验 4（上）：FM vs DDPM 受控对比——同一网络、同一训练步数、只改路径与目标；样本质量随采样步数 {2, 5, 20, 100} 的变化，3 个种子 | 2.5 小时 |
| B | 研究实验 4（下）：MNIST 类别条件生成，guidance 强度 $w \in \{0, 1, 3\}$（Ho & Salimans 约定）：判为目标数字的比例、类别熵与多样性的权衡，含纯噪声阴性对照 | 2 小时 |
| C | 实验记录（正式提交格式） | 1 小时 |
| D | Capstone 主实验推进 | 1 小时 |
| E | 代码讲解（本周有口头抽查） | 0.5 小时 |

**规则**：可以使用 AI 编程助手写代码，但必须完成 E 部分的"代码讲解"（用自己的话解释关键代码，并注明哪些部分由 AI 生成）。**本周是口头代码讲解周**，每人 5–10 分钟，从 E 部分问起。

**评分**（研究实验评分表）：设计 30% · 执行 30% · 分析与解释 30% · 表达 10%。

**运行时间**：A 在 CPU 上约半分钟到一分钟；B 在 CPU 上约 3–4 分钟（Colab GPU 约半分钟）。为了在 CPU 上也能跑完，B 用 14×14 的 MNIST、训练 1500 步；有 GPU 的同学可以按 B 部分末尾的提示放大。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision.utils import make_grid
from zero2researcher import data, experiments, generative
from zero2researcher.checks import week17 as ck

# A 部分（2D 小网络）固定用 CPU；B 部分（MNIST U-Net）自动选择：Colab 上是 CUDA，Apple 芯片是 MPS，否则 CPU
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| B 部分的 device:", device)

## A · 研究实验 4（上）：Flow Matching vs DDPM

**研究问题**：同一个网络、同一份训练预算下，Flow Matching（线性路径 + 速度目标 + 欧拉采样）与 DDPM（方差保持路径 + 噪声目标 + 祖先采样）的样本质量随采样步数怎样变化？少步时谁更好？差距是否大于种子间的波动？

实验规格：

- **自变量**：方法 ∈ {FM, DDPM}（训练里只差两行：概率路径与预测目标；采样器随方法而定）× 采样步数 $n \in \{2, 5, 20, 100\}$；
- **数据**：工具包的月牙与双高斯（默认参数）；
- **固定**：网络 3→128→128→2（SiLU）、Adam 1e-3、batch 512、训练 3000 步；DDPM 用连续调度 $\bar\alpha(\tau) = \exp(-(0.1\tau + 9.95\tau^2))$；
- **种子** 0、1、2：**同一个种子下 FM 与 DDPM 的初始化、训练数据流完全相同**——只有那两行不同；每个 (数据集, 方法, 种子) 训练一个模型，四种步数共用；采样噪声也由同一个种子决定；
- **评估**：每次生成 2000 个点，与固定的真实参考样本（2000 个点，`rng=2026`）比较能量距离与 k-NN precision；报告均值 ± 标准差，并给出噪声地板。

**A0 · 预注册**（跑任何代码之前写下，跑完不许改）：你预期哪种方法在少步（2、5 步）时更好？100 步时呢？DDPM 的种子间波动会比 FM 大还是小？什么样的结果会推翻你的预期？

- 预期：
- 什么结果会推翻它：

下面的零件第 16 周写过，原样给出：

In [ ]:
class VelocityMLP(nn.Module):
    """输入 (x, t) 的 3→128→128→2 MLP；FM 里输出速度，DDPM 里输出噪声 ε"""

    def __init__(self, hidden=128, depth=2):
        super().__init__()
        layers, d = [], 3
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.SiLU()]
            d = hidden
        layers.append(nn.Linear(d, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        t = torch.as_tensor(t, dtype=x.dtype).reshape(-1, 1).expand(len(x), 1)
        return self.net(torch.cat([x, t], dim=1))


def euler_integrate(u, x0, n_steps, t_start=0.0, t_end=1.0):
    h = (t_end - t_start) / n_steps
    x, path = x0, [x0]
    for i in range(n_steps):
        x = x + h * u(x, t_start + i * h)
        path.append(x)
    return torch.stack(path)


def noise(shape, seed=0):
    # 在 CPU 上用固定种子抽标准正态噪声：换设备也得到同一份噪声
    return torch.randn(shape, generator=torch.Generator().manual_seed(seed))


DATA = {"moons": generative.sample_moons, "gaussian mixture": generative.sample_gaussian_mixture}
REFS = {name: sample(2000, rng=2026) for name, sample in DATA.items()}
for name, sample in DATA.items():
    eds = [generative.energy_distance(sample(2000, rng=s), REFS[name]) for s in range(10)]
    precs = [generative.knn_precision(sample(2000, rng=s), REFS[name]) for s in range(10)]
    print(f"{name:17s} 噪声地板（10 次真 vs 真）：能量距离 {np.mean(eds):+.4f} ± {np.std(eds, ddof=1):.4f}"
          f" | precision {np.mean(precs):.3f} ± {np.std(precs, ddof=1):.3f}")

**A1** · 写 `alpha_bar(tau)`（讲义 S1 / 课堂练习 E1）。

In [ ]:
def alpha_bar(tau):
    # 在这里写你的代码
    raise NotImplementedError


ck.alpha_bar(alpha_bar)

**A2** · `train_2d(sample_data, kind, seed, steps=3000)` 的框架已给出，补上 `kind` 决定的那两行（讲义 S1 的对照表）：FM 用直线路径、目标是速度；DDPM 用 $\bar\alpha(t)$ 的方差保持路径、目标是噪声。

In [ ]:
def train_2d(sample_data, kind, seed, steps=3000, batch_size=512):
    experiments.set_seed(seed)
    model = VelocityMLP()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(steps):
        x_data = sample_data(batch_size)
        eps = torch.randn_like(x_data)
        t = torch.rand(batch_size, 1)
        if kind == "FM":
            # 在这里写你的代码
            raise NotImplementedError
        else:
            # 在这里写你的代码
            raise NotImplementedError
        loss = ((model(xt, t) - target) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model

**A3** · 写两个采样器，都从 `noise((n, 2), seed)` 出发、只返回终点：

- `fm_sample(model, n, n_steps, seed)`：欧拉法从 $t = 0$ 走到 $t = 1$；
- `ddpm_sample(eps_model, n, n_steps, seed)`：讲义 S1 的跨步祖先采样（$\tau$ 网格 1 → 0，$\alpha = \bar\alpha_{\text{当前}}/\bar\alpha_{\text{下一个}}$，$\sigma^2 = \frac{1-\bar\alpha_{\text{下一个}}}{1-\bar\alpha_{\text{当前}}}(1 - \alpha)$）；每步加的新噪声用 `torch.Generator().manual_seed(seed + 1)` 抽。

两个都在 `torch.no_grad()` 下运行。

In [ ]:
@torch.no_grad()
def fm_sample(model, n, n_steps, seed):
    # 在这里写你的代码
    raise NotImplementedError


@torch.no_grad()
def ddpm_sample(eps_model, n, n_steps, seed):
    # 在这里写你的代码
    raise NotImplementedError


print("冒烟测试:", tuple(fm_sample(VelocityMLP(), 4, 2, 0).shape), tuple(ddpm_sample(VelocityMLP(), 4, 2, 0).shape))

**A4** · 主循环。要求的结构（自查会检查）：

```python
results[数据集名][方法][步数] = {"ed": [3 个种子], "precision": [3 个种子]}     # 方法 ∈ {"FM", "DDPM"}
```

DDPM 的 2 步采样会把误差放大几十倍，个别种子的能量距离可能很大——这是要报告的结果，不是 bug。

In [ ]:
STEPS, SEEDS, METHODS = (2, 5, 20, 100), (0, 1, 2), ("FM", "DDPM")
results = {name: {m: {n: {"ed": [], "precision": []} for n in STEPS} for m in METHODS} for name in DATA}
t0 = time.time()
# 在这里写你的代码
raise NotImplementedError
print(f"{len(DATA) * len(METHODS) * len(SEEDS)} 个模型的训练与评估共用时 {time.time() - t0:.0f} 秒")
ck.fm_vs_ddpm_results(results)

**A5** · 结果表 `table`：行 = 步数；列 = 每个数据集 × 方法 × 指标，格子里写 `均值 ± 标准差`。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
table

**A6** · 误差棒图：2 行（数据集）× 2 列（能量距离、precision），横轴采样步数（对数刻度），FM 与 DDPM 各一条线，误差棒 = 标准差；能量距离的纵轴用 `symlog`（`linthresh=1e-3`）。标题写结论。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**A7 · 分析问题**（每题两三句话，引用你表里的数字）

1. 在每个步数上，FM 与 DDPM 谁更好？差距是否大于两者的标准差（第 6 周三分法）？结论在两份数据上一致吗？
2. 2 步时两种方法的种子间标准差各是多少？如果只跑了一个种子，你可能得出什么错误结论？
3. 本实验里"方法"其实捆绑了三样东西：路径、预测目标、采样器。你的结果能说明是**哪一样**造成了少步时的差距吗？设计一个能把它们分开的后续实验（提示：讲义 S2 里"FM 训练的模型放进 DDPM 采样器"的演示；工具包的 `convert_prediction`）。
4. 本实验的固定超参数（MLP 大小、3000 步、DDPM 的线性 β 调度、σ² 的取法）里，哪一个最可能对 DDPM 不公平？你的结论要加什么限定条件？

（在这里回答）

## B · 研究实验 4（下）：guidance 强度的权衡

**研究问题**：在 MNIST 类别条件生成中，classifier-free guidance 的强度 $w$（Ho & Salimans 约定，$w = 0$ 为不引导）如何影响"生成的数字被认成目标数字的比例"与生成的多样性？

实验规格：

- **模型**：工具包 `SmallUNet(img_size=14, num_classes=10)`（约 17 万参数；编号 10 是 null 类别），Flow Matching 训练，以 $p_{\text{uncond}} = 0.1$ 把类别换成 null；AdamW 2e-3、batch 128、1500 步；数据 = 14×14 MNIST 训练集的 10000 张；
- **自变量**：guidance 强度 $w \in \{0, 1, 3\}$——**只训练一个模型**，三种 $w$ 都用它采样，且用**同一份起始噪声**；
- **采样**：每个数字 20 张（共 200 张），欧拉 50 步；
- **指标**：分类器（`generative.train_mnist_classifier`）判为目标数字的比例、平均置信度、预测类别直方图的归一化熵、多样性（同一目标数字的 20 张图两两像素 L2 距离的平均，`generative.within_class_diversity`）；
- **对照**：真实测试数字 200 张（参考上限）；**纯噪声阴性对照**——把采样器的起始噪声直接交给分类器。

**B0 · 预注册**：三个指标会随 $w$ 怎样变化？纯噪声那一行你预期是什么样子？

- 预期：
- 什么结果会推翻它：

In [ ]:
x_train, y_train = generative.load_mnist(size=14, train=True, n=10000, seed=0)
P_UNCOND, STEPS_COND, BATCH = 0.1, 1500, 128


def show_grid(images, nrow=10, title="", ax=None):
    grid = make_grid(images.detach().cpu().clamp(-1, 1), nrow=nrow, normalize=True, value_range=(-1, 1))
    ax = ax or plt.figure(figsize=(4.5, 4.5)).gca()
    ax.imshow(grid.permute(1, 2, 0))
    ax.axis("off")
    ax.set_title(title)

**B1** · 训练循环：每步随机抽 128 张图与标签、搬到 `device`；以概率 `P_UNCOND` 把标签换成 `model.null_class`（`torch.where`）；FM 的插值与速度目标（时间形状 `(B, 1, 1, 1)`）；`model(x_t, t, y)`；五步核心。

In [ ]:
experiments.set_seed(0)
model = generative.SmallUNet(img_size=14, num_classes=10).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
losses = []
t0 = time.time()
model.train()
for step in range(STEPS_COND):
    # 在这里写你的代码
    raise NotImplementedError
    losses.append(loss.item())
    if (step + 1) % 500 == 0:
        print(f"step {step + 1:5d} | 最近 100 步平均损失 {np.mean(losses[-100:]):.4f} | 已用 {time.time() - t0:.0f} 秒")
model.eval()
print(f"训练用时 {time.time() - t0:.0f} 秒")

**B2** · 写 `cfg_combine`（Ho 约定）与 `guided_sample(model, targets, w, n_steps=50, seed=1)`：起始噪声 `noise((len(targets), 1, 14, 14), seed)` 搬到 `device`；每一步的速度 = `cfg_combine(model(x, t, y), model(x, t, y_null), w)`，其中 `y_null` 全是 `model.null_class`；用 `euler_integrate` 从 0 积分到 1；返回终点（搬回 CPU）。

In [ ]:
def cfg_combine(pred_cond, pred_uncond, w):
    # 在这里写你的代码
    raise NotImplementedError


@torch.no_grad()
def guided_sample(model, targets, w, n_steps=50, seed=1):
    # 在这里写你的代码
    raise NotImplementedError


ck.cfg_combine(cfg_combine)

**B3** · 评估：训练分类器；对每个 $w$ 用 `guided_sample` 生成 200 张（每个数字 20 张），把结果存进

```python
guidance[w] = {"frac_target": ..., "mean_confidence": ..., "class_entropy": ..., "diversity": ...}
samples[w] = 生成的 200 张图
```

前三个来自 `generative.classifier_scores(clf, 图, targets)`，多样性来自 `generative.within_class_diversity(图, targets)`。

In [ ]:
clf = generative.train_mnist_classifier(size=14)
targets = torch.arange(10).repeat_interleave(20)
guidance, samples = {}, {}
# 在这里写你的代码
raise NotImplementedError
for w, r in guidance.items():
    print(f"w = {w}: " + " | ".join(f"{k} {v:.3f}" for k, v in r.items()))
ck.guidance_tradeoff(guidance)

**B4** · 两个对照行：

- `controls["pure noise"]`：起始噪声本身（`noise((200, 1, 14, 14), seed=1)`，与采样器的起点完全相同），目标类别同 `targets`；
- `controls["real digits"]`：`generative.load_mnist(size=14, train=False, n=200, seed=3)` 的真实测试数字，目标类别用它们的真实标签。

每行同样存四个指标。再算出纯噪声被判成**最常见的那个类别**的比例 `frac_top`（= `max(pred_counts) / 200`），连同纯噪声的类别熵交给 `ck.noise_control`。

In [ ]:
controls = {}
# 在这里写你的代码
raise NotImplementedError
ck.noise_control(frac_top, controls["pure noise"]["class_entropy"])

**B5** · 汇总表与权衡图（代码已给出，跑通即可）：

In [ ]:
summary = pd.DataFrame({**{"real digits": controls["real digits"], "pure noise": controls["pure noise"]},
                        **{f"w = {w}": r for w, r in guidance.items()}}).T.round(3)
display(summary)

fig, axes = plt.subplots(1, 4, figsize=(17, 4.4))
ax = axes[0]
ax.plot([guidance[w]["diversity"] for w in guidance], [guidance[w]["frac_target"] for w in guidance], "o-", color="C0")
for w in guidance:
    ax.annotate(f"w = {w}", (guidance[w]["diversity"], guidance[w]["frac_target"]),
                textcoords="offset points", xytext=(5, -12))
ax.scatter([controls["real digits"]["diversity"]], [controls["real digits"]["frac_target"]], color="k", marker="*",
           s=120, label="Real digits")
ax.set_xlabel("Within-class diversity (mean pairwise L2)")
ax.set_ylabel("Fraction classified as the requested digit")
ax.set_title("Guidance trades diversity for on-target rate")
ax.legend(loc="lower left")
for ax, w in zip(axes[1:], (0, 1, 3)):
    show_grid(samples[w].view(10, 20, 1, 14, 14)[:, :10].reshape(100, 1, 14, 14), nrow=10, title=f"w = {w}", ax=ax)
plt.tight_layout()
plt.show()

**B6 · 分析问题**（每题两三句话，引用你表里的数字）

1. 随 $w$ 增大，三个指标各怎样变化？多样性降到了真实数字的多少？$w = 3$ 的样本"更好"吗——好在哪里，差在哪里？
2. 纯噪声那一行说明了什么？如果某位同学只报告"$w = 0$ 时，要求生成 8 的样本有 X% 被认成 8"，你会怎样质疑？为什么"判为目标的比例"必须和类别熵、多样性、阴性对照一起报告？
3. 我们只训练了一个模型、只用了一份起始噪声。这对"$w$ 的效应"的结论意味着什么（哪些随机性被控制了、哪些没有被评估）？如果再跑 3 个训练种子，你预计哪个指标的波动最大？
4. Ho & Salimans 原文 Table 1–2（64×64 / 128×128 ImageNet）中，最好的 FID 出现在很小的引导（64×64 上 $w = 0.1$，128×128 上 $w = 0.3$），最好的 Inception Score 出现在强引导（$w \ge 4$，扫描范围的上端）。结合你的结果解释为什么两个指标会选出不同的 $w$。
5. 如果你用的是 MIT 讲义的约定，这三组实验分别对应 $w_{\text{MIT}}$ 等于多少？

（在这里回答）

## C · 实验记录（研究实验 4，正式提交）

把 A 与 B 写成**一条**实验记录（两部分共享研究主题："只改路径与目标"的受控对比，以及 guidance 这一个旋钮）。"结果"一栏写均值 ± 标准差并引用 A5、A6 的表和图与 B5 的汇总表；"局限"一栏必须讨论：A 中"方法"捆绑了路径 / 目标 / 采样器；B 中只有一个训练种子；以及纯噪声对照揭示的分类器指标的局限。

## 实验记录 #__ · 日期：____

- 研究问题：
- 假设（预期结果，以及什么结果会推翻它）：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；哪些干扰变量单独调优）：
- 基线：
- 数据与划分方式（如何避免泄漏）：
- 随机种子：
- 结果（数值：均值±标准差；图：链接或插图）：
- 解释：
- 局限：
- 下一步：

## D · Capstone 主实验推进

第 18 周要交主结果（≥ 3 个种子）。用几行写下本周的进展：

- 本周完成的主实验运行（条件 × 种子）：
- 每次运行都在实验记录本里有条目吗？
- 本周发现的问题与解决办法：
- 你的 Capstone 的"阴性对照"或"平凡基线"是什么（参考 B4）？如果还没有，打算怎么设计？

## E · 代码讲解（本周口头抽查）

用你自己的话写一段（约 150–300 字）：

1. A 部分的 `train_2d`：FM 与 DDPM 那两行各是什么意思？为什么同一个种子下两种方法的初始化和训练数据完全相同——这对结论有什么意义？`ddpm_sample` 里传给模型的时间是 FM 的 $t$ 还是 DDPM 的 $\tau$，为什么？
2. B 部分的训练循环与 `guided_sample`：null 类别是怎么实现的？$w = 0$ 时 `cfg_combine` 返回什么？为什么三种 $w$ 要用同一份起始噪声？
3. 注明哪些代码由 AI 助手生成、你做了哪些检查或修改。

讲不清楚的代码视同不是你写的——口头抽查就从这一段问起。

（在这里写）